# Open Source Text Embeddings

In [1]:
from utils.llm import get_embedding_model

embed_model = get_embedding_model(model="BAAI/bge-m3")

c:\Users\arkan\Downloads\University\TA\ta-rag-demo\env\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
from utils.utils import load_from_pickle

all_documents = load_from_pickle("all_documents")

In [7]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    all_documents, embed_model, collection_name="uu", path="qdrant"
)

In [9]:
retriever = vectorstore.as_retriever()
retriever.invoke("What are the provinces of South Sumatera")

[Document(metadata={'title': 'Undang-undang Nomor 17 Tahun 2022 Tentang Provinsi Sumatera Barat', 'jenis_bentuk_peraturan': 'UNDANG-UNDANG', 'pemrakarsa': 'PEMERINTAH PUSAT', 'nomor': 17, 'tahun': 2022, 'tentang': 'Provinsi Sumatera Barat', 'tempat_penetapan': 'Jakarta', 'ditetapkan_tanggal': '25 Juli 2022', 'status': 'Berlaku', 'dokumen_peraturan': 'https://peraturan.go.id/files/uu17-2022.pdf', 'id': 'cayt2uju', 'filename': 'uu_25_juli_2022_uu17_2022_cayt2uju.pdf', 'page_number': 3, '_id': 'a7b905de8c204d558a3329182dc43d57', '_collection_name': 'uu'}, page_content='DOC NAME: Undang-undang Nomor 17 Tahun 2022 Tentang Provinsi Sumatera Barat | UNDANG-UNDANG\n\n\n- 3 -\nBAB II\nCAKUPAN WILAYAH, IBU KOTA, DAN KARAKTERISTIK\nPROVINSI SUMATERA BARAT\nPasal 3\n(1) Provinsi Sumatera Barat terdiri atas 12 (dua belas)\nkabupaten dan 7 (tujuh) kota, yaitu:\na. Kabupaten Pesisir Selatan;\nb. Kabupaten Solok;.\nc. Kabupaten Sijunjung;\nd. Kabupaten Tanah Datar;\ne. Kabupaten Padang Pariaman;\nf. K

In [13]:
from langchain_community.chat_models import ChatOllama

llm =ChatOllama(model="llama3.1", temperature=0.1)
llm.invoke("What are the provinces of South Sumatera")

AIMessage(content='The province of South Sumatra (also known as Sumatera Selatan) in Indonesia is divided into 12 regencies and 2 cities. Here are the names of the provinces:\n\n1. Banyuasin Regency\n2. Lahat Regency\n3. Musi Banyuasin Regency\n4. Musi Rawas Regency\n5. Ogan Ilir Regency\n6. Ogan Komering Ulu Regency\n7. Ogan Komering Ulu Selatan Regency\n8. Prabumulih City\n9. Penukal Abung Regency\n10. Pesawaran Regency\n11. South Tengah Regency\n12. Tanjung Jabung Barat Regency\n13. Tanjung Jabung Timur Regency', response_metadata={'model': 'llama3.1', 'created_at': '2024-08-05T15:41:36.8208801Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 3123565000, 'load_duration': 14573500, 'prompt_eval_count': 19, 'prompt_eval_duration': 241298000, 'eval_count': 167, 'eval_duration': 2866674000}, id='run-a5fd3984-16d5-4e12-8aa4-c9d89a677980-0')

# Trying out QdrantIndexManager

In [4]:
def clean_document_content(content):
    return content.replace('\x00', '')  # Remove NUL characters

In [3]:
from utils.llm import get_embedding_model
from utils.utils import load_from_pickle


embed_model = get_embedding_model(model="BAAI/bge-m3")
all_documents = load_from_pickle("all_documents")

In [5]:
from utils.document.split import document_splitter

all_splits = document_splitter(docs=all_documents)
all_splits_sorted = sorted(all_splits, key=lambda x: (x.metadata['id'], x.metadata.get('page_number', '0')))
for split in all_splits_sorted:
    split.page_content = clean_document_content(split.page_content)

Split 4050 documents into 13052 chunks


In [22]:
from utils.database.qdrant import QdrantIndexManager

qdrant_index_manager = QdrantIndexManager(embed_model, index_name="uu")

In [26]:
qdrant_index_manager.store_vector_index(all_splits, batch_size=200)

Loaded 1-200 documents
Loaded 201-400 documents
Loaded 401-600 documents
Loaded 601-800 documents
Loaded 801-1000 documents
Loaded 1001-1200 documents
Loaded 1201-1400 documents
Loaded 1401-1600 documents
Loaded 1601-1800 documents
Loaded 1801-2000 documents
Loaded 2001-2200 documents
Loaded 2201-2400 documents
Loaded 2401-2600 documents
Loaded 2601-2800 documents
Loaded 2801-3000 documents
Loaded 3001-3200 documents
Loaded 3201-3400 documents
Loaded 3401-3600 documents
Loaded 3601-3800 documents
Loaded 3801-4000 documents
Loaded 4001-4200 documents
Loaded 4201-4400 documents
Loaded 4401-4600 documents
Loaded 4601-4800 documents
Loaded 4801-5000 documents
Loaded 5001-5200 documents
Loaded 5201-5400 documents
Loaded 5401-5600 documents
Loaded 5601-5800 documents
Loaded 5801-6000 documents
Loaded 6001-6200 documents
Loaded 6201-6400 documents
Loaded 6401-6600 documents
Loaded 6601-6800 documents
Loaded 6801-7000 documents
Loaded 7001-7200 documents
Loaded 7201-7400 documents
Loaded 7401-

In [29]:
vector_store = qdrant_index_manager.load_vector_index()

Loaded index 'uu'.


In [31]:
if vector_store:
    results = vector_store.similarity_search("Apa itu capital Jawa Barat?", k=50)
    for result in results:
        print(result.page_content, result.metadata)

NEPUEUT INDONE;IA
-4-
t. Kota Sukabumi;
u. Kota Bandung;
v. Kota Cirebon;
w. Kota Bekasi;
x. Kota Depok;
y. Kota Cimahi;
z. Kota Tasikmalaya; dan
aa. Kota Banjar.
Pasal 4
Ibu kota Provinsi Jawa Barat berkedudukan di Kota
Bandung.
Pasal 5
Provinsi Jawa Barat memiliki karakteristik, yaitu:
a.
dengan ciri geogralis utama kawasan
dataran rendah berupa pesisir dan pantai, kawasan
dataran tinggi berupa pegunungan dan
kawasan taman nasional serta kawasan lindung dan
konservasi yang merupakan bagian dari potensi
kewilayahan Provinsi Jawa Barat;
b. potensi sumber daya alam berupa pertanian,
kehutanan, kelautan dan perikanan, pertambangan,
potensi pariwisata, potensi perdagangan, potensi
industri, potensi ekonomi kreatif, dan potensi lainnya;
dan
c. suku bangsa dan kultural yang secara umum memiliki
karakter religius dan berbudaya sekaligus menjunjung
tinegi adat istiadat dan kelestarian lingkungan.
BABIII ...
SK No l8ll04A

Source: [Undang-undang Nomor 10 Tahun 2023 Tentang Provinsi Jawa Barat]

In [25]:
# qdrant_index_manager.delete_index()

Deleted index 'uu'.
